# Assignment 4: Diffusion Model for Image Generation

## Introduction

Introduction of Assignment 4 can be found at: https://docs.google.com/presentation/d/13hljrI3GVDEp8EPtoJoNWBT8EtIBhd0TxLUZkryNH5Q/edit?usp=sharing.

## Meta Instructions
1. Environment: Install pytorch >= 1.7.0 and torchvision >= 0.8.0 to avoid any issues caused by package versions.

2. Finish coding tasks according to instructions in this file. **Only** change the code within the regions marked by "Your code starts here" and "Your code ends here". Do not modify anywhere else including comments. This task does not require GPUs and can meet the basic requirements.

3. Please use the trained diffusion model to generate an image that contains the string of number in your StuID (e.g., if your StuID is A0123456J, please generate 0123456) and save it as a single jpg file, it is encouraged that you train the diffusion model by yourself. If you cannot find the computing resources, you can try to use this pretrained model for generating locally. (https://drive.google.com/file/d/1-9dozojlZxdpkei5lyxFeKofKYq4rQIf/view?usp=sharing)

4. Submission: submit files including code implementation "StuID_Assignment_4.ipynb", one-page report "StuID_Assignment_4.pdf" and generated image "StuID_Assignment_4.jpg". The submission deadline is **23:59 on Mar 27**.


### Optional Puzzles for Extra Credits

You are able to get full marks of this assignment by correctly completing and submitting the code and PDF report according to the above requirements. However, if you want to earn 1-2 bonus points in your total assignment grade (not exceeding the maximum score), you can choose to solve one of the following optional puzzles and include it in your final codes and reports:

1. **Puzzle 1 (Metric Design)**: The FID score (Fréchet Inception Distance) is a popular metric for evaluating generative models, but it has well-known failure modes. Design a new evaluation metric for generative models that addresses at least one specific limitation of FID. You need to: (a) provide a clear mathematical definition, (b) prove or argue why it addresses the limitation, (c) implement it and compare against FID on your trained model's outputs under different guidance weights. *(Hint: consider FID's Gaussian assumption, its conflation of quality and diversity, or its sensitivity to sample size.)*

2. **Puzzle 2 (Open Question: Further Improvement)**: How to further improve the performance of your model, under various metrics?


# Diffusion Model for Image Generation

Diffusion Models, including the Denoising Diffusion Probabilistic Model (DDPM), represent a powerful class of generative models that simulate the gradual process of diffusing data into noise and then denoising it back into coherent samples.

Inspired by the natural diffusion process observed in physics, DDPM operates in two phases: a forward phase where data is incrementally noised until it becomes indistinguishable from random noise, and a reverse phase where this process is inverted, gradually denoising to generate new data samples that closely mimic the original data distribution.

This innovative approach allows DDPM to produce high-quality, diverse outputs across various domains such as images, audio, and text, without the adversarial training complexities associated with other generative models.

**References:**
- Denoising Diffusion Probabilistic Models (DDPM) Paper:  [DDPM](https://arxiv.org/abs/2006.11239)
- Blog about Diffusion models: [DDPM Blog](https://lilianweng.github.io/posts/2021-07-11-diffusion-models/)
- Improved DDPM: [Improved Denoising Diffusion Probabilistic Models](https://arxiv.org/abs/2102.09672)

This assignment requires you to implement a DDPM-based diffusion model for generating handwritten digits images (32×32).

## Import Packages

It is to import the necessary packages.

In [ ]:
from typing import Dict, Tuple
from tqdm import tqdm
import os
os.makedirs('./data/diffusion_outputs10', exist_ok=True)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
from torchvision import models, transforms
from torchvision.datasets import MNIST
from torchvision.utils import save_image, make_grid
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
import numpy as np

## Create dataset

Load the MNIST dataset and apply the following preprocessing:

Resize images to image_size×image_size (you can use the torchvision.transforms.Resize function).
Normalize pixel values to the range [-1, 1].

In [ ]:
def create_mnist_dataloaders(batch_size, image_size=32, num_workers=4):
    
    preprocess = transforms.Compose([
        transforms.Resize(image_size),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])

    train_dataset = torchvision.datasets.MNIST(
        root="./mnist_data",
        train=True,
        download=True,
        transform=preprocess
    )
    
    test_dataset = torchvision.datasets.MNIST(
        root="./mnist_data", 
        train=False,
        download=True,
        transform=preprocess
    )

    return DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers),\
           DataLoader(test_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)



# Model Architecture: U-Net

Build a neural network to predict noise. You can use a U-Net or any convolutional architecture of your choice. The network should take as input the noisy image along with the time-step information and output the predicted noise. Make sure that:

- The network handles MNIST images (e.g., 32×32).

- The network has a bottleneck structure with multiple downsampling and upsampling layers.

- The network includes residual connections.



In [ ]:
class ResidualConvBlock(nn.Module):
    def __init__(
        self, in_channels: int, out_channels: int, is_res: bool = False
    ) -> None:
        super().__init__()
        self.same_channels = in_channels==out_channels
        self.is_res = is_res
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, 1, 1),
            nn.BatchNorm2d(out_channels),
            nn.GELU(),
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, 3, 1, 1),
            nn.BatchNorm2d(out_channels),
            nn.GELU(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.is_res:
            out = self.conv1(x)
            out = self.conv2(out)
            if self.same_channels:
                return x + out
            else:
                return out
        else:
            out = self.conv1(x)
            out = self.conv2(out)
            return out

class UnetDown(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(UnetDown, self).__init__()
        self.model = nn.Sequential(
            ResidualConvBlock(in_channels, out_channels),
            nn.MaxPool2d(2)
        )
    def forward(self, x):
        return self.model(x)

class UnetUp(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(UnetUp, self).__init__()
        self.model = nn.Sequential(
            nn.ConvTranspose2d(in_channels, out_channels, 2, 2),
            ResidualConvBlock(out_channels, out_channels),
            ResidualConvBlock(out_channels, out_channels),
        )
    def forward(self, x, skip):
        x = torch.cat((x, skip), 1)
        return self.model(x)

class EmbedFC(nn.Module):
    def __init__(self, input_dim, emb_dim):
        super(EmbedFC, self).__init__()
        self.input_dim = input_dim
        layers = [
            nn.Linear(input_dim, emb_dim),
            nn.GELU(),
            nn.Linear(emb_dim, emb_dim),
        ]
        self.model = nn.Sequential(*layers)
    def forward(self, x):
        x = x.view(-1, self.input_dim)
        return self.model(x)

class ContextUnet(nn.Module):
    def __init__(self, in_channels, n_feat = 256, n_classes=10):
        super(ContextUnet, self).__init__()
        self.in_channels = in_channels
        self.n_feat = n_feat
        self.n_classes = n_classes
        self.init_conv = ResidualConvBlock(in_channels, n_feat, is_res=True)
        self.down1 = UnetDown(n_feat, n_feat)
        self.down2 = UnetDown(n_feat, 2 * n_feat)
        self.to_vec = nn.Sequential(nn.AvgPool2d(7), nn.GELU())
        self.timeembed1 = EmbedFC(1, 2*n_feat)
        self.timeembed2 = EmbedFC(1, 1*n_feat)
        self.contextembed1 = EmbedFC(n_classes, 2*n_feat)
        self.contextembed2 = EmbedFC(n_classes, 1*n_feat)
        self.up0 = nn.Sequential(
            nn.ConvTranspose2d(2 * n_feat, 2 * n_feat, 8, 8),
            nn.GroupNorm(8, 2 * n_feat),
            nn.ReLU(),
        )
        self.up1 = UnetUp(4 * n_feat, n_feat)
        self.up2 = UnetUp(2 * n_feat, n_feat)
        self.out = nn.Sequential(
            nn.Conv2d(2 * n_feat, n_feat, 3, 1, 1),
            nn.GroupNorm(8, n_feat),
            nn.ReLU(),
            nn.Conv2d(n_feat, self.in_channels, 3, 1, 1),
        )

    def forward(self, x, c, t, context_mask):
        x = self.init_conv(x)
        down1 = self.down1(x)
        down2 = self.down2(down1)
        hiddenvec = self.to_vec(down2)
        c = nn.functional.one_hot(c, num_classes=self.n_classes).type(torch.float)
        context_mask = context_mask[:, None]
        context_mask = context_mask.repeat(1,self.n_classes)
        context_mask = (-1*(1-context_mask))
        c = c * context_mask
        cemb1 = self.contextembed1(c).view(-1, self.n_feat * 2, 1, 1)
        temb1 = self.timeembed1(t).view(-1, self.n_feat * 2, 1, 1)
        cemb2 = self.contextembed2(c).view(-1, self.n_feat, 1, 1)
        temb2 = self.timeembed2(t).view(-1, self.n_feat, 1, 1)
        up1 = self.up0(hiddenvec)
        up2 = self.up1(cemb1*up1+ temb1, down2)
        up3 = self.up2(cemb2*up2+ temb2, down1)
        out = self.out(torch.cat((up3, x), 1))
        return out


## The Diffusion Model

### Noise Schedules

The noise schedule $\{\beta_t\}_{t=1}^T$ controls how quickly noise is added during the forward process. The original DDPM uses a **linear schedule**:

$$\beta_t = \beta_1 + \frac{t-1}{T-1}(\beta_T - \beta_1)$$

However, [Nichol & Dhariwal (2021)](https://arxiv.org/abs/2102.09672) proposed a **cosine schedule** that provides a more gradual noising process, which often leads to better generation quality:

$$\bar{\alpha}_t = \frac{f(t)}{f(0)}, \quad f(t) = \cos\left(\frac{t/T + s}{1+s} \cdot \frac{\pi}{2}\right)^2$$

where $s = 0.008$ is a small offset to prevent $\beta_t$ from being too small near $t=0$.

**Task**: Implement both the linear and cosine noise schedules.

In [ ]:
def ddpm_schedules(beta1, beta2, T, schedule_type='linear'):
    if schedule_type == 'linear':
        beta_t = (beta2 - beta1) * torch.arange(0, T + 1, dtype=torch.float32) / T + beta1
        sqrt_beta_t = torch.sqrt(beta_t)
        alpha_t = 1 - beta_t
        alphabar_t = torch.cumsum(torch.log(alpha_t), dim=0).exp()
    elif schedule_type == 'cosine':
        s = 0.008
        steps = T + 1
        x = torch.linspace(0, T, steps)
        alphas_cumprod = torch.cos(((x / T) + s) / (1 + s) * torch.pi / 2)**2
        alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
        alphas = alphas_cumprod[1:] / alphas_cumprod[:-1]
        alphas = torch.cat([torch.ones(1), alphas])
        beta_t = 1 - alphas
        beta_t = torch.clamp(beta_t, 0, 0.999)
        sqrt_beta_t = torch.sqrt(beta_t)
        alphabar_t = alphas_cumprod
        alpha_t = alphas
    
    sqrtab = torch.sqrt(alphabar_t)
    oneover_sqrta = 1 / torch.sqrt(alpha_t)
    sqrtmab = torch.sqrt(1 - alphabar_t)
    mab_over_sqrtmab_inv = (1 - alpha_t) / sqrtmab

    return {
        "alpha_t": alpha_t,
        "oneover_sqrta": oneover_sqrta,
        "sqrt_beta_t": sqrt_beta_t,
        "alphabar_t": alphabar_t,
        "sqrtab": sqrtab,
        "sqrtmab": sqrtmab,
        "mab_over_sqrtmab": mab_over_sqrtmab_inv,
    }


### Visualize the Noise Schedules

**Task**: Plot $\bar{\alpha}_t$ for both the linear and cosine schedules on the same figure. Observe the difference in how quickly the signal is destroyed.

In [ ]:
T = 400
linear_sched = ddpm_schedules(1e-4, 0.02, T, 'linear')
cosine_sched = ddpm_schedules(1e-4, 0.02, T, 'cosine')
plt.figure(figsize=(10, 6))
plt.plot(linear_sched['alphabar_t'].numpy(), label='Linear')
plt.plot(cosine_sched['alphabar_t'].numpy(), label='Cosine')
plt.xlabel('Timestep t')
plt.ylabel('Alphabar_t')
plt.title('Comparison of Noise Schedules')
plt.legend()
plt.grid(True)
plt.savefig('./data/diffusion_outputs10/schedule_comparison.png')
plt.show()


### Forward and Reverse Diffusion Process

Implement the noise addition process (forward) and the denoising process (reverse).

The forward process adds noise:
$$x_t = \sqrt{\bar{\alpha}_t} \cdot x_0 + \sqrt{1 - \bar{\alpha}_t} \cdot \epsilon, \quad \epsilon \sim \mathcal{N}(0, I)$$

The reverse process denoises:
$$x_{t-1} = \frac{1}{\sqrt{\alpha_t}} \left( x_t - \frac{1-\alpha_t}{\sqrt{1-\bar{\alpha}_t}} \epsilon_\theta(x_t, t) \right) + \sqrt{\beta_t} \cdot z, \quad z \sim \mathcal{N}(0, I)$$

In [ ]:
class DDPM(nn.Module):
    def __init__(self, nn_model, betas, n_T, device, drop_prob=0.1, schedule_type='linear'):
        super(DDPM, self).__init__()
        self.nn_model = nn_model.to(device)
        for k, v in ddpm_schedules(betas[0], betas[1], n_T, schedule_type).items():
            self.register_buffer(k, v)
        self.n_T = n_T
        self.device = device
        self.drop_prob = drop_prob
        self.loss_mse = nn.MSELoss()

    def forward(self, x, c):
        _ts = torch.randint(1, self.n_T+1, (x.shape[0],)).to(self.device)
        noise = torch.randn_like(x)
        sqrtab = self.sqrtab[_ts].view(-1, 1, 1, 1)
        sqrtmab = self.sqrtmab[_ts].view(-1, 1, 1, 1)
        x_t = sqrtab * x + sqrtmab * noise
        context_mask = torch.bernoulli(torch.zeros_like(c)+self.drop_prob).to(self.device)
        return self.loss_mse(noise, self.nn_model(x_t, c, _ts / self.n_T, context_mask))

    def sample(self, n_sample, size, device, guide_w = 0.0):
        x_i = torch.randn(n_sample, *size).to(device)
        c_i = torch.arange(0,10).to(device)
        c_i = c_i.repeat(int(n_sample/c_i.shape[0]))
        context_mask = torch.zeros_like(c_i).to(device)
        c_i = c_i.repeat(2)
        context_mask = context_mask.repeat(2)
        context_mask[n_sample:] = 1.
        x_i_store = []
        for i in range(self.n_T, 0, -1):
            t_is = torch.tensor([i / self.n_T]).to(device).view(-1, 1, 1, 1).repeat(n_sample, 1, 1, 1)
            x_in = x_i.repeat(2, 1, 1, 1)
            t_in = t_is.repeat(2, 1, 1, 1)
            z = torch.randn(n_sample, *size).to(device) if i > 1 else 0
            eps = self.nn_model(x_in, c_i, t_in, context_mask)
            eps1 = eps[:n_sample]
            eps2 = eps[n_sample:]
            eps = (1+guide_w)*eps1 - guide_w*eps2
            oneover_sqrta = self.oneover_sqrta[i]
            mab_over_sqrtmab = self.mab_over_sqrtmab[i]
            sqrt_beta_t = self.sqrt_beta_t[i]
            x_i = oneover_sqrta * (x_i - mab_over_sqrtmab * eps) + sqrt_beta_t * z
            if i%20==0 or i==self.n_T or i<8:
                x_i_store.append(x_i.detach().cpu().numpy())
        return x_i, np.array(x_i_store)


## DDIM Sampler (Accelerated Sampling)

The standard DDPM sampler requires iterating through all $T$ timesteps, which is slow. [DDIM (Denoising Diffusion Implicit Models)](https://arxiv.org/abs/2010.02502) enables faster sampling by using a non-Markovian process that skips timesteps.

Given a subsequence of timesteps $\tau_1 < \tau_2 < \ldots < \tau_S$ where $S \ll T$, the DDIM update rule is:

$$x_{\tau_{i-1}} = \sqrt{\bar{\alpha}_{\tau_{i-1}}} \underbrace{\left( \frac{x_{\tau_i} - \sqrt{1-\bar{\alpha}_{\tau_i}} \cdot \epsilon_\theta(x_{\tau_i}, \tau_i)}{\sqrt{\bar{\alpha}_{\tau_i}}} \right)}_{\text{predicted } x_0} + \sqrt{1-\bar{\alpha}_{\tau_{i-1}} - \sigma^2_{\tau_i}} \cdot \epsilon_\theta(x_{\tau_i}, \tau_i) + \sigma_{\tau_i} \cdot z$$

When $\sigma_{\tau_i} = 0$, sampling becomes **deterministic** (DDIM). When $\sigma_{\tau_i} = \sqrt{\frac{1-\bar{\alpha}_{\tau_{i-1}}}{1-\bar{\alpha}_{\tau_i}}} \cdot \sqrt{\beta_{\tau_i}}$, it reduces to DDPM.

**Task**: Implement DDIM sampling as an additional method of the DDPM class.

In [ ]:
def ddim_sample(self, n_sample, size, device, guide_w=0.0, ddim_steps=50, eta=0.0):
    device = torch.device(device)
    n_T = self.n_T
    times = torch.linspace(1, n_T, ddim_steps).long().to(device)
    times_prev = torch.cat([torch.tensor([0]).to(device), times[:-1]])
    x_i = torch.randn(n_sample, *size).to(device)
    c_i = torch.arange(0, 10).to(device)
    c_i = c_i.repeat(int(n_sample/c_i.shape[0]))
    x_i_store = []
    for i in range(ddim_steps - 1, -1, -1):
        t_idx = times[i]
        t_prev_idx = times_prev[i]
        t_is = (torch.ones(n_sample) * t_idx / n_T).to(device).view(-1, 1, 1, 1)
        x_in = x_i.repeat(2, 1, 1, 1)
        t_in = t_is.repeat(2, 1, 1, 1)
        c_in = c_i.repeat(2)
        m_in = torch.zeros_like(c_in).to(device)
        m_in[n_sample:] = 1.
        eps_all = self.nn_model(x_in, c_in, t_in, m_in)
        eps1 = eps_all[:n_sample]
        eps2 = eps_all[n_sample:]
        eps = (1 + guide_w) * eps1 - guide_w * eps2
        ab_t = self.alphabar_t[t_idx]
        ab_prev = self.alphabar_t[t_prev_idx]
        pred_x0 = (x_i - torch.sqrt(1 - ab_t) * eps) / torch.sqrt(ab_t)
        if eta > 0:
            sigma = eta * torch.sqrt((1 - ab_prev) / (1 - ab_t) * (1 - ab_t / ab_prev))
        else:
            sigma = 0
        dir_xt = torch.sqrt(1 - ab_prev - sigma**2) * eps
        noise = torch.randn_like(x_i) if i > 0 else 0
        x_i = torch.sqrt(ab_prev) * pred_x0 + dir_xt + sigma * noise
        if i % 10 == 0 or i == ddim_steps - 1:
            x_i_store.append(x_i.detach().cpu().numpy())
    return x_i, np.array(x_i_store)
DDPM.ddim_sample = ddim_sample


## Trainer

Including the main function.

You should complete the inference code that uses the trained model to generate desired images.

In [ ]:
def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def train_mnist(schedule_type='linear'):

    imagesize = 32
    # hardcoding these here
    n_epoch = 20
    batch_size = 256
    n_T = 400 # 500
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    n_classes = 10
    n_feat = 128 # 128 ok, 256 better (but slower)
    lrate = 1e-4
    save_model = False
    save_dir = './data/diffusion_outputs10/'
    ws_test = [0.0, 0.5, 2.0] # strength of generative guidance

    ddpm = DDPM(nn_model=ContextUnet(in_channels=1, n_feat=n_feat, n_classes=n_classes), betas=(1e-4, 0.02), n_T=n_T, device=device, drop_prob=0.1, schedule_type=schedule_type)
    ddpm.to(device)

    # optionally load a model
    # ddpm.load_state_dict(torch.load("./data/diffusion_outputs/ddpm_unet01_mnist_9.pth"))

    dataloader, _ = create_mnist_dataloaders(batch_size=batch_size, image_size=imagesize, num_workers=4)
    optim = torch.optim.Adam(ddpm.parameters(), lr=lrate)

    for ep in range(n_epoch):
        print(f'epoch {ep}')
        ddpm.train()

        # linear lrate decay
        optim.param_groups[0]['lr'] = lrate*(1-ep/n_epoch)

        pbar = tqdm(dataloader)
        loss_ema = None
        for x, c in pbar:
            optim.zero_grad()
            x = x.to(device)
            c = c.to(device)
            loss = ddpm(x, c)
            loss.backward()
            if loss_ema is None:
                loss_ema = loss.item()
            else:
                loss_ema = 0.95 * loss_ema + 0.05 * loss.item()
            pbar.set_description(f"loss: {loss_ema:.4f}")
            optim.step()
        
        # for eval, save an image of currently generated samples (top rows)
        # followed by real images (bottom rows)
        ddpm.eval()
        with torch.no_grad():
            n_sample = 4*n_classes
            for w_i, w in enumerate(ws_test):
                x_gen, x_gen_store = ddpm.sample(n_sample, (1, 32, 32), device, guide_w=w)

                # append some real images at bottom, order by class also
                x_real = torch.Tensor(x_gen.shape).to(device)
                for k in range(n_classes):
                    for j in range(int(n_sample/n_classes)):
                        try: 
                            idx = torch.squeeze((c == k).nonzero())[j]
                        except:
                            idx = 0
                        x_real[k+(j*n_classes)] = x[idx]

                x_all = torch.cat([x_gen, x_real])
                grid = make_grid(x_all*-1 + 1, nrow=10)
                save_image(grid, save_dir + f"image_ep{ep}_w{w}.png")
                print('saved image at ' + save_dir + f"image_ep{ep}_w{w}.png")

                if ep%5==0 or ep == int(n_epoch-1):
                    # create gif of images evolving over time, based on x_gen_store
                    fig, axs = plt.subplots(nrows=int(n_sample/n_classes), ncols=n_classes,sharex=True,sharey=True,figsize=(8,3))
                    def animate_diff(i, x_gen_store):
                        print(f'gif animating frame {i} of {x_gen_store.shape[0]}', end='\r')
                        plots = []
                        for row in range(int(n_sample/n_classes)):
                            for col in range(n_classes):
                                axs[row, col].clear()
                                axs[row, col].set_xticks([])
                                axs[row, col].set_yticks([])
                                # plots.append(axs[row, col].imshow(x_gen_store[i,(row*n_classes)+col,0],cmap='gray'))
                                plots.append(axs[row, col].imshow(-x_gen_store[i,(row*n_classes)+col,0],cmap='gray',vmin=(-x_gen_store[i]).min(), vmax=(-x_gen_store[i]).max()))
                        return plots
                    ani = FuncAnimation(fig, animate_diff, fargs=[x_gen_store],  interval=200, blit=False, repeat=True, frames=x_gen_store.shape[0])    
                    ani.save(save_dir + f"gif_ep{ep}_w{w}.gif", dpi=100, writer=PillowWriter(fps=5))
                    print('saved image at ' + save_dir + f"gif_ep{ep}_w{w}.gif")
        # optionally save model
        if save_model and ep == int(n_epoch-1):
            torch.save(ddpm.state_dict(), save_dir + f"model_{ep}.pth")
            print('saved model at ' + save_dir + f"model_{ep}.pth")

In [ ]:
def generate_samples(model_path, save_dir, n_samples=40, image_size=(1, 32, 32), device="cuda"):
    device = torch.device(device)
    n_classes = 10
    n_feat = 64
    n_T = 400
    model = DDPM(nn_model=ContextUnet(in_channels=1, n_feat=n_feat, n_classes=n_classes), 
                 betas=(1e-4, 0.02), n_T=n_T, device=device, drop_prob=0.1)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()
    stuid_digits = [0, 1, 2, 3, 4, 5, 6]
    n_sample = len(stuid_digits)
    with torch.no_grad():
        x_i = torch.randn(n_sample, *image_size).to(device)
        c_i = torch.tensor(stuid_digits).to(device)
        for i in range(model.n_T, 0, -1):
            t_is = (torch.ones(n_sample) * i / model.n_T).to(device).view(-1, 1, 1, 1)
            z = torch.randn(n_sample, *image_size).to(device) if i > 1 else 0
            guide_w = 2.0
            x_in = x_i.repeat(2, 1, 1, 1)
            t_in = t_is.repeat(2, 1, 1, 1)
            c_in = c_i.repeat(2)
            m_in = torch.zeros_like(c_in).to(device)
            m_in[n_sample:] = 1.
            eps_all = model.nn_model(x_in, c_in, t_in, m_in)
            eps1 = eps_all[:n_sample]
            eps2 = eps_all[n_sample:]
            eps = (1 + guide_w) * eps1 - guide_w * eps2
            oneover_sqrta = model.oneover_sqrta[i]
            mab_over_sqrtmab = model.mab_over_sqrtmab[i]
            sqrt_beta_t = model.sqrt_beta_t[i]
            x_i = oneover_sqrta * (x_i - mab_over_sqrtmab * eps) + sqrt_beta_t * z
        grid = make_grid(x_i*-1 + 1, nrow=len(stuid_digits))
        save_image(grid, save_dir + "A0123456J_Manus_Assignment_4.jpg")
        print(f"Generated StuID image at {save_dir}A0123456J_Manus_Assignment_4.jpg")


In [ ]:
if __name__ == "__main__":
    set_seed()
    print("Starting Linear Schedule Training...")
    train_mnist(schedule_type='linear')
    print("Starting Cosine Schedule Training...")
    train_mnist(schedule_type='cosine')
    # Use the model from the last epoch of cosine training
    generate_samples('./data/diffusion_outputs10/model_1.pth', './data/diffusion_outputs10/', device='cpu')


## Comparison Experiment

**Task**: Train two models — one with the **linear** schedule and one with the **cosine** schedule. Compare their generation quality visually and report your observations.

Additionally, compare the standard DDPM sampler (400 steps) with your DDIM sampler at different step counts (e.g., 20, 50, 100 steps). For each configuration, save a grid of generated images.

In your report, discuss:
1. Which noise schedule produces better results and why?
2. How does DDIM sample quality change as you reduce the number of steps?
3. At what step count does DDIM quality become noticeably worse than full DDPM?

In [ ]:
################################
# Your code starts here
# Run comparison experiments and save results
################################



################################
# Your code ends here
################################